### Adsorption energies


Adsorption energies measure how strongly molecules bind to surfaces. The adsorption energy is:

$H_{ads} = E_{slab+adsorbate} - E_{slab} - E_{adsorbate}$

A negative value indicates favorable (exothermic) adsorption.


#### Clean Pt(111) slab calculation


In [1]:
from vasp import Vasp
from ase.lattice.surface import fcc111, add_adsorbate
from ase.constraints import FixAtoms

# Clean Pt(111) slab - small size for speed
slab = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
constraint = FixAtoms(mask=[atom.tag > 1 for atom in slab])
slab.set_constraint(constraint)

calc_slab = Vasp(label='surfaces/Pt-slab',
                 xc='PBE',
                 encut=300,
                 kpts=(3, 3, 1),
                 ibrion=2,
                 nsw=10,
                 isif=0,
                 atoms=slab)

e_slab = calc_slab.get_potential_energy()
print(f'Clean slab energy: {e_slab:.4f} eV')

Clean slab energy: -68.2362 eV


/opt/conda/lib/python3.9/site-packages/ase/lattice/surface.py:17: UserWarning: Moved to ase.build
  warnings.warn('Moved to ase.build')


#### O atom on fcc site


In [2]:
# O at fcc site
slab_fcc = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
add_adsorbate(slab_fcc, 'O', height=1.2, position='fcc')
constraint = FixAtoms(mask=[atom.symbol != 'O' and atom.tag > 1 for atom in slab_fcc])
slab_fcc.set_constraint(constraint)

calc_fcc = Vasp(label='surfaces/Pt-slab-O-fcc',
                xc='PBE',
                encut=300,
                kpts=(3, 3, 1),
                ibrion=2,
                nsw=20,
                isif=0,
                atoms=slab_fcc)

e_fcc = calc_fcc.get_potential_energy()
print(f'O@fcc energy: {e_fcc:.4f} eV')

O@fcc energy: -74.2302 eV


#### O atom on hcp site


In [3]:
# O at hcp site
slab_hcp = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
add_adsorbate(slab_hcp, 'O', height=1.2, position='hcp')
constraint = FixAtoms(mask=[atom.symbol != 'O' and atom.tag > 1 for atom in slab_hcp])
slab_hcp.set_constraint(constraint)

calc_hcp = Vasp(label='surfaces/Pt-slab-O-hcp',
                xc='PBE',
                encut=300,
                kpts=(3, 3, 1),
                ibrion=2,
                nsw=20,
                isif=0,
                atoms=slab_hcp)

e_hcp = calc_hcp.get_potential_energy()
print(f'O@hcp energy: {e_hcp:.4f} eV')

O@hcp energy: -74.5386 eV


#### O atom on bridge site


In [4]:
# O at bridge site
slab_bridge = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
add_adsorbate(slab_bridge, 'O', height=1.2, position='bridge')
constraint = FixAtoms(mask=[atom.symbol != 'O' and atom.tag > 1 for atom in slab_bridge])
slab_bridge.set_constraint(constraint)

calc_bridge = Vasp(label='surfaces/Pt-slab-O-bridge',
                   xc='PBE',
                   encut=300,
                   kpts=(3, 3, 1),
                   ibrion=2,
                   nsw=20,
                   isif=0,
                   atoms=slab_bridge)

e_bridge = calc_bridge.get_potential_energy()
print(f'O@bridge energy: {e_bridge:.4f} eV')

O@bridge energy: -74.9810 eV


#### O2 molecule reference


In [5]:
from ase import Atoms

# O2 molecule in a box
o2 = Atoms('O2', positions=[[0, 0, 0], [0, 0, 1.21]])
o2.center(vacuum=5.0)

calc_o2 = Vasp(label='molecules/O2',
               xc='PBE',
               encut=300,
               kpts=(1, 1, 1),
               ismear=0,
               sigma=0.01,
               ibrion=2,
               nsw=10,
               atoms=o2)

e_o2 = calc_o2.get_potential_energy()
print(f'O2 energy: {e_o2:.4f} eV')

O2 energy: -8.7816 eV


#### Adsorption energy analysis


In [6]:
# Calculate adsorption energies
# Using 1/2 O2 as reference
e_o = e_o2 / 2

H_fcc = e_fcc - e_slab - e_o
H_hcp = e_hcp - e_slab - e_o
H_bridge = e_bridge - e_slab - e_o

print('Adsorption energies (eV/O atom):')
print(f'  fcc site:    {H_fcc:.3f} eV')
print(f'  hcp site:    {H_hcp:.3f} eV')
print(f'  bridge site: {H_bridge:.3f} eV')
print()
print('The most stable site has the most negative adsorption energy.')
if H_fcc < H_hcp and H_fcc < H_bridge:
    print('FCC site is most stable.')
elif H_hcp < H_fcc and H_hcp < H_bridge:
    print('HCP site is most stable.')
else:
    print('Bridge site is most stable (unusual for O on Pt).')

Adsorption energies (eV/O atom):
  fcc site:    -1.603 eV
  hcp site:    -1.912 eV
  bridge site: -2.354 eV

The most stable site has the most negative adsorption energy.
Bridge site is most stable (unusual for O on Pt).
